# MATH 440 — Final Project
## Model Selection from Curated Datasets

**Name:** [Your name]  
**Date submitted:** [Date]  
**Dataset:** [DS number and scenario title]

---
> **Before submitting:**
> - Replace every `[placeholder]` with your own content.
> - The notebook must run top-to-bottom without errors: Kernel → Restart and Run All.
> - Delete all scratch cells.
> - Filename: `LastName_FirstName_FinalProject.ipynb`

---
## Imports and Global Settings

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit

# Use Numpy's modern RNG (for the bootstrap in Step 5)
rng = np.random.default_rng(2026)

# Golden-ratio figure dimensions (inches)
golden_ratio = (1 + 5**0.5) / 2
FIG_HEIGHT   = 5
FIG_WIDTH    = FIG_HEIGHT * golden_ratio

# Specify font size and figure quality for all figures
plt.rcParams.update({'font.size': 11, 'figure.dpi': 300})

## Data and Configuration
Define your data file path and any fixed parameter values here.

In [3]:
# Load your data
DATA_FILE = 'DS14.csv' # update to your assigned dataset

# Fixed parameters from scenario card (examples). Leave as None if not used.
THETA_FIXED = 0.7
C_FIXED     = None

# Load the dataset using pandas
df = pd.read_csv(DATA_FILE)
print("Column names: ", df.columns.tolist()) # shows the actual column names
print(df.head()) # quick sanity check

t_data = df.iloc[:, 0].to_numpy(dtype=float)   # column 0 = time
y_data = df.iloc[:, 1].to_numpy(dtype=float)   # column 1 = QOI
n_obs  = len(y_data)

print(f'Dataset: {DATA_FILE}')
print(f'Observations: n = {n_obs}')
print(f'Time range: {t_data[0]:.4g} to {t_data[-1]:.4g}')
print(f'QOI range: {y_data.min():.4g} to {y_data.max():.4g}')
print(f'Initial value: N(0) ≈ {y_data[0]:.4g}')

Column names:  ['year', 'cohort_biomass_mt']
   year  cohort_biomass_mt
0     0              138.0
1     1              246.0
2     2              528.0
3     3              940.0
4     4              913.0
Dataset: DS14.csv
Observations: n = 55
Time range: 0 to 54
QOI range: 138 to 1.194e+04
Initial value: N(0) ≈ 138


---
## Step 2. Analyze (Raw data plot)

In [4]:
# Raw Data Plot

# plt.figure(figsize=(FIG_WIDTH, FIG_HEIGHT))
# plt.scatter(XXX, XXX, s=20, color='steelblue', label='Observed data')
# plt.xlabel(XXX)   
# plt.ylabel(XXX)             
# plt.title(XXX)  
# plt.legend(XXX)
# plt.tight_layout()
# plt.savefig('XXX.png')
# plt.show()

---
## Step 3. Simulate and Fit

### How to write a model function

`curve_fit` calls your function as `f(t, param1, param2, ...)` and
expects an array of predicted values back.  The **logistic model** is used
in all three examples below so you can compare the patterns directly.
Write the closed-form solution whenever it exists.

**Pattern 1: Closed-form analytical solution (logistic):**
```python
def model_logistic_analytical(t, r, K):
    # N(t) = K / (1 + ((K - N0)/N0) * exp(-r*t))
    return K / (1 + ((K - N0) / N0) * np.exp(-r * t))
```

**Pattern 2: Continuous ODE via solve_ivp (same logistic, no closed form assumed):**
```python
def model_logistic_ode(t, r, K):
    def rhs(s, N):
        return [r * N[0] * (1 - N[0] / K)]   # dN/dt = r*N*(1 - N/K)
    sol = solve_ivp(rhs, [t[0], t[-1]], [N0], t_eval=t)
    return sol.y[0]
```

**Pattern 3: Discrete map via for-loop (discrete logistic):**
```python
def model_logistic_discrete(t, r, K):
    N = np.zeros(len(t))
    N[0] = N0
    for i in range(len(t) - 1):
        N[i+1] = N[i] * (1 + r * (1 - N[i] / K))
    return N
```

### Key lessons for fitting

1. Initial guess for $K$: use `y_data.max() * 1.5`, not `y_data.max()`.
   If the data have not yet reached the plateau, `max(data)` underestimates $K$
   and the optimizer can get stuck.
2. Initial guess for $N(0)$: use `y_data[0]` (the actual first observed value).
3. Bounds enforce physical constraints: carrying capacity must exceed the
   observed maximum; rates must be positive.
4. `maxfev=10000` gives the optimizer enough iterations for models
   that are expensive to evaluate (ODE-based or discrete).

### Model functions

In [5]:
# Pattern 1: Closed-form analytical solution (use this whenever it exists)
# Logistic ODE: dN/dt = r*N*(1 - N/K)
# Analytical solution: N(t) = K / (1 + A*exp(-r*t))
def _example_closed_form(t, r, K):
    A = (K - N0) / N0      # A is a derived constant, not the population variable
    return K / (1 + A * np.exp(-r * t))

# Pattern 2: Continuous ODE with no closed form (use solve_ivp)
# Use this when your model has no analytical solution (e.g., Smith, theta-logistic).
# Logistic ODE: dN/dt = r*N*(1 - N/K) (same model as Pattern 1, now solved numerically)
def _example_ode(t, r, K):
    def rhs(s, N): # N is the state vector [population] passed by solve_ivp
        return [r * N[0] * (1 - N[0] / K)]
    sol = solve_ivp(rhs, [t[0], t[-1]], [N0], t_eval=t)
    return sol.y[0]

# Pattern 3: Discrete map (iterate a for-loop; no solve_ivp needed)
# Discrete logistic: N_{t+1} = N_t * (1 + r*(1 - N_t/K))
def _example_discrete(t, r, K):
    N = np.zeros(len(t)) # N is the population array, one entry per time step
    N[0] = N0
    for i in range(len(t) - 1):
        N[i+1] = N[i] * (1 + r * (1 - N[i] / K))
    return N

In [6]:
# Model 1: Logistic
def model1_logistic(t, r, K):
    """
    Analytical solution of the continuous logistic ODE:
    • dN/dt = r*N*(1 - N/K)
    • N(t) = K / (1 + ((K - N0)/N0) * exp(-r*t))

    Parameters:
    • r:  intrinsic growth rate per unit time (units: 1/time) 
    • K:  carrying capacity (same units as QOI)
    """
    return K / (1 + ((K - N0) / N0) * np.exp(-r * t))

# Model 2: [Name]
def model2_[name](t, r, K):
    """
    [1-2 line description of the model and its ODE or recurrence relation.]

    Parameters:
    • r: [description and units]
    • K: [description and units]
    """
    pass

# Model 3: [Name]
def model3_[name](t, r, K):
    """
    [Description. Parameters r and K with units.]
    """
    pass

# Model 4: [Name]
def model4_[name](t, r, K):
    """
    [Description. Parameters r and K with units.]
    """
    pass

# Model 5: [Name]
def model5_[name](t, r, K):
    """
    [Description. Parameters r and K with units.]
    """
    pass

### Fit Model 1: Logistic

In [7]:
# WRITE YOUR COMMENT HERE
p0_1 = [0.1, y_data.max() * 1.5] # [r_guess, K_guess]

bounds_1 = (
    [0, y_data.max() * 1.01], # lower: r > 0, K > max(data)
    [20, 1e7],                # upper: physically reasonable
)

params_1, cov_1 = curve_fit(
    model1_logistic, t_data, y_data,
    p0=p0_1,
    bounds=bounds_1,
    maxfev=10000,
)
perr_1 = np.sqrt(np.diag(cov_1))
ypred_1 = model1_logistic(t_data, *params_1)

print('Model 1: Logistic')
print(f'  r = {params_1[0]:.5g} ± {perr_1[0]:.5g}')
print(f'  K = {params_1[1]:.5g} ± {perr_1[1]:.5g}')


NameError: name 'N0' is not defined

### Figure 2: Super Plot

### Figure 3: Top Candidate Model

---
## Step 4. Validate and Select


### Figure 4: Residual Plots

In [ ]:
# Compute and plot residuals for Model 1: [NAME]
resid_1 = y_data - ypred_1

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_WIDTH))

ax.scatter(XXX, XXX, s=12)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Model 1:[NAME]')
ax.set_ylabel('Residual')
ax.set_title('Residuals: observed − predicted', fontsize=13)
plt.tight_layout()
plt.savefig('YOUR CODE HERE', bbox_inches='tight')
plt.show()

### AICc values

$$
\mathrm{AIC} = n\ln\!\left(\frac{\mathrm{RSS}}{n}\right) + 2k,
\qquad
\mathrm{AIC}_c = \mathrm{AIC} + \frac{2k(k+1)}{n-k-1},
\qquad
\Delta = \mathrm{AIC}_c^{(i)} - \min_j\,\mathrm{AIC}_c^{(j)} \ge 0
$$

$k=$ number of **fitted** parameters only.  Fixed parameters printed as numbers on your scenario card do **not** count.


In [ ]:
k = XXX

def compute_aicc(y_obs, y_hat, k, n):
    rss = # YOUR CODE HERE
    aic = # YOUR CODE HERE
    aicc = # YOUR CODE HERE
    return rss, aicc

rss_1, aicc_1 = compute_aicc(# YOUR CODE HERE)
rss_5, aicc_5 = compute_aicc(# YOUR CODE HERE)

best_aicc = min(# YOUR CODE HERE)

print(f'n = {n_obs},  k = {k}')
print()
print(f'{"Model":<26} {"RSS":>12}  {"AICc":>10}  {"Δ":>8}')
print('─' * 62)
print(f'{"Model 1: Logistic":<26} {rss_1:>12.4f}  {aicc_1:>10.3f}  {aicc_1 - best_aicc:>8.3f}')
print(f'{"Model 2: [Name]":<26} {rss_2:>12.4f}  {aicc_2:>10.3f}  {aicc_2 - best_aicc:>8.3f}')
print(f'{"Model 3: [Name]":<26} {rss_3:>12.4f}  {aicc_3:>10.3f}  {aicc_3 - best_aicc:>8.3f}')
print(f'{"Model 4: [Name]":<26} {rss_4:>12.4f}  {aicc_4:>10.3f}  {aicc_4 - best_aicc:>8.3f}')
print(f'{"Model 5: [Name]":<26} {rss_5:>12.4f}  {aicc_5:>10.3f}  {aicc_5 - best_aicc:>8.3f}')


SyntaxError: invalid syntax (4243165955.py, line 4)

### Figure 5: Best-fitting model

In [ ]:
# Set these three variables to your winning model before running.
params_w = params_1 # update
perr_w   = perr_1   # update
ypred_w  = ypred_1  # update
label_w  = 'Model 1: [NAME]' # update

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

### Figure 6: Holdout MSE (80/20 split)

---
## Step 5. Interpret and Forecast

> Parameter interpretation and the Limitations discussion belong entirely
> in your **LaTeX report** (§ Step 5).  No code is required here for those
> subsections.
>
> This notebook section covers the **Forecast** (Step 5a, bonus) and the
> **AI Audit** model fit (Step 6, optional code).


### Step 5a: Parametric Bootstrap Forecast  *(bonus, +5 pts)*

The forecast simulates the winning model for 12 additional time steps
beyond the last observed data point.  The 95 % confidence interval is
built by drawing 1,000 alternative parameter vectors from the multivariate
normal distribution described by `params_w` and `cov_w`, propagating each
vector forward 12 steps, and taking the 2.5th and 97.5th percentiles at
each future step.

In [ ]:
# Step 5: Forecasting with the winning model
winning_model = model1_XXX  # update to your winning model function
cov_w = cov_XXX  # update to the matching covariance matrix

N_FORECAST = 12
N_BOOTSTRAP = 1000

# Future time grid
dt = t_data[1] - t_data[0]
t_future = t_data[-1] + dt * np.arange(1, N_FORECAST + 1)

# Generate 1,000 alternative parameter vectors
# Each row of `samples` is a parameter vector drawn from the multivariate normal distribution
# This captures both the uncertainty in the parameter estimates and any correlations between parameters
samples = rng.multivariate_normal(params_w, cov_w, size=N_BOOTSTRAP)

# Simulate each draw N_FORECAST forward
boot = np.full((N_BOOTSTRAP, N_FORECAST), np.nan)
for i in range(N_BOOTSTRAP):
    boot[i] = winning_model(t_future, *samples[i])

# 95% CI
lower = np.nanpercentile(boot, 2.5, axis=0)
upper = np.nanpercentile(boot, 97.5, axis=0)

y_fit = winning_model(t_data, *params_w)
y_forecast = winning_model(t_future, *params_w)

# Figure 7: 12-Step Forecast with 95% CI
fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIGURE_HEIGHT))

ax.scatter(t_data, y_data, s=20, color='black', label='Observed data', zorder=5)
ax.plot(t_data, y_fit,  color='steelblue', lw=2, label=f'{label_w} (fitted)')
ax.plot(t_future, y_forecast, color='steelblue', lw=2, linestyle='--',
        label='12-step forecast')
ax.fill_between(t_future, lower, upper, color='steelblue', alpha=0.25,
                label='95% CI')
ax.axvline(t_data[-1], color='gray', lw=1, linestyle=':', label='End of data')

ax.set_xlabel('[time variable and units]')
ax.set_ylabel('[QOI and units]')
ax.set_title(f'12-Step Forecast: {label_w}')
ax.legend()
plt.tight_layout()
plt.savefig('step5_forecast.png', bbox_inches='tight')
plt.show()

# Summary statistics
obs_range  = y_data.max() - y_data.min()
band_width = upper[-1] - lower[-1]

print(f'Forecast at step 12:')
print(f'  Predicted          : {y_forecast[-1]:.5g}')
print(f'  95% CI             : [{lower[-1]:.5g},  {upper[-1]:.5g}]')
print(f'  CI width           : {band_width:.5g}')
print(f'  Observed data range: {obs_range:.5g}')
print(f'  Band / range       : {100 * band_width / obs_range:.1f} %')
print('  (> 20–30% = poorly constrained forecast)')

### Step 6: AI Audit *(required; code optional)*

> The four-item audit paragraph belongs in your **LaTeX report** (§AI Audit),
> and the full Gemini exchange goes in Appendix A.
>
> If you chose to fit the model Gemini proposed, implement it in the cell
> below using the same pattern as your five main models.


In [ ]:
# Optional: fit the AI-proposed model
# If Gemini proposed an additional model and you pursued it, implement it
# here. Report its AICc and Δ relative to the winning model in your report.

# def model_gemini(t, param1, param2):
#     """
#     [Gemini-proposed model name]: [brief equation description]
#     """
#     pass


---
## Bonus: Holdout MSE  *(+5 pts)*

Each model is refit on the first 80% of time points (the **training** set).
Mean squared error is then computed on the withheld final 20% (the
**test** set, also called the holdout set because the model has never seen
those points). A model that overfits the training data will show a
noticeably higher holdout MSE than a model that generalises well.

**Important caveat for saturating models.** If your dataset is DS01–DS10
(continuous-time, monotone), the final 20% may contain the plateau region
that carries the most information about the carrying capacity *K*. In that
case, a high holdout MSE may reflect the absence of plateau data in the
training set, not overfitting (acknowledge this in your report if it
applies).

In [ ]:
# Holdout split
n_train = int(0.8 * n_obs)
t_train = t_data[:n_train]
y_train = y_data[:n_train]
t_test = t_data[n_train:]
y_test = y_data[n_train:]

print(f'Training set : {n_train} points  '
      f'({100 * n_train / n_obs:.0f} %  of data)')
print(f'Test set     : {n_obs - n_train} points  '
      f'({100 * (n_obs - n_train) / n_obs:.0f} %  of data)')
print()

# Fit each model on training data; evaluate on test data
print(f'{"Model":<14}  {"Holdout MSE":>14}')
print('─' * 32)


In [ ]:
# Holdout visualisation for the winning model
# Shows training data, test data, and the model curve fitted on training only,
# with a vertical dashed line at the 80/20 boundary.

params_tr_w, mse_w = holdout_mse[WINNING_MODEL]
y_full_tr = winning_model_func(t_data, *params_tr_w)

fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, FIG_HEIGHT))
ax.scatter(t_train, y_train, s=20, color='steelblue',
            label=f'Training data (first {n_train} pts, 80%)')
ax.scatter(t_test,  y_test,  s=20, color='darkorange',
            label=f'Test data (last {n_obs - n_train} pts, 20%)',
            zorder=5)
ax.plot(t_data, y_full_tr, color='steelblue', lw=2.0,
        label=f'{WINNING_MODEL} (fitted on training)')
ax.axvline(t_data[n_train - 1], color='gray', lw=1.2,
            linestyle='--', label='80/20 split')
ax.set_xlabel('[Time variable and units]')
ax.set_ylabel('[QOI and units]')
ax.set_title(f'Holdout Validation: {WINNING_MODEL}')
ax.legend()
plt.tight_layout()
plt.savefig('step4_holdout.png', bbox_inches='tight')
plt.show()
print('Saved: step4_holdout.png')
print(f'Holdout MSE for {WINNING_MODEL}: {mse_w:.4f}')